# Vector Database Data Ingestion 

In [1]:
import os
from pathlib import Path
from typing import Any, override
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader

load_dotenv(Path("../.env.local"))

from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

In [2]:
DATA_DIR = Path("../data")
COLLECTION_NAME = "salon_info"
QDRANT_URL = (os.getenv("QDRANT_URL") or "").strip()
QDRANT_API_KEY = (os.getenv("QDRANT_API_KEY") or "").strip() or None
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

if not QDRANT_URL:
    raise ValueError(
        "QDRANT_URL is not set. Run the cell above (load_dotenv) first, "
        "and ensure .env.local or .env contains QDRANT_URL=..."
    )

# Load Documents

In [3]:
loader = TextLoader(DATA_DIR / "data.txt")
documents = loader.load()

# Split documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = text_splitter.split_documents(documents)


In [4]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
client = QdrantClient(url = QDRANT_URL, api_key = QDRANT_API_KEY)

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
    )

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)

vector_store.add_documents(chunks)
print(f"Added {len(chunks)} documents to Qdrant")


Added 10 documents to Qdrant


# Database ingestion

In [57]:
import sqlite3

conn = sqlite3.connect("../data/salon.db")
cursor = conn.cursor()

In [21]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS services (
    name TEXT PRIMARY KEY,
    duration INTEGER
)
""")

In [22]:
cursor.executemany("""
INSERT OR REPLACE INTO services (name, duration)
VALUES (?, ?)
""", [
    ("Women's Haircut", 45),
    ("Men's Haircut", 30),
    ("Children's Haircut (under 12)", 25),
    ("Root Touch-Up", 60),
    ("Full Hair Coloring", 90),
    ("Highlights", 120),
    ("Balayage", 150),
    ("Blow-Dry", 30),
    ("Wash & Style", 40),
    ("Formal Updo", 60),
    ("Bridal Hairstyling", 90),
    ("Deep Conditioning Treatment", 30),
    ("Keratin Treatment", 120),
    ("Scalp Treatment", 20),
])

In [23]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS business_hours (
    day TEXT PRIMARY KEY,
    open_time TEXT,
    close_time TEXT
)
""")

In [58]:
cursor.executemany("""
INSERT OR REPLACE INTO business_hours (day, open_time, close_time)
VALUES (?, ?, ?)
""", [
    ("monday", 540, 1080),
    ("tuesday", 540, 1080),
    ("wednesday", 540, 1080),
    ("thursday", 540, 1080),
    ("friday", 540, 1080),
    ("saturday", 540, 960)
])

In [25]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS bookings (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    phone TEXT,
    service TEXT,
    date TEXT,
    start_minutes INTEGER,
    end_minutes INTEGER
)
""")


In [59]:
conn.commit()   
conn.close() 

In [55]:
conn = sqlite3.connect("../data/salon.db")
cursor = conn.cursor()
cursor.execute("SELECT * FROM bookings")
rows = cursor.fetchall()

for row in rows:
    print(row)

conn.close()

(24, 'Karla Udiljak', '385987654321', "Women's Haircut + Blow-Dry", '2026-03-19', 660, 735, 'elhv81bu5vlpa0bd1v96psli7s')
